# SR830 XY-only frequency and amplitude sweeps

Read-only analysis of completed frequency and excitation-amplitude sweeps. XX is discarded at the loader boundary. Every figure labels the XY harmonic order, for example `XY · h1`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

from attodry_control.commissioning_analysis import (
    browse_commissioning_file,
    discover_commissioning_records,
    summarize_commissioning_file,
)
from attodry_control.xy_sweep_analysis import (
    load_xy_sweep_samples,
    plot_xy_sweep,
    xy_sweep_harmonic,
)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
DATA_DIRECTORY

## Select completed frequency and amplitude records

The newest completed frequency and excitation records are selected automatically. Set `OPEN_BROWSER=True` to replace the matching scan with a file selected through the Windows Browse dialog.

In [ ]:
OPEN_BROWSER = False
selected_path = (
    browse_commissioning_file(DATA_DIRECTORY) if OPEN_BROWSER else None
)
completed_catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses={'completed'},
    scan_types={'frequency', 'excitation'},
)
frequency_record = next(
    (item for item in completed_catalog if item.scan_type == 'frequency'), None
)
excitation_record = next(
    (item for item in completed_catalog if item.scan_type == 'excitation'), None
)
if frequency_record is None or excitation_record is None:
    raise FileNotFoundError('A completed frequency or excitation record is missing.')
frequency_path = frequency_record.path
excitation_path = excitation_record.path
if selected_path is not None:
    selected_summary = summarize_commissioning_file(selected_path)
    if selected_summary.scan_type == 'frequency':
        frequency_path = selected_path
    elif selected_summary.scan_type == 'excitation':
        excitation_path = selected_path
    else:
        raise ValueError('Browse selection must be a frequency or excitation sweep.')
frequency_path, excitation_path

## Load clean XY samples only

XX is not returned. Change `SAMPLE_STATUSES` only for explicit status audit work; rejected files additionally require `INCLUDE_REJECTED=True`.

In [ ]:
SAMPLE_STATUSES = {'clean'}
INCLUDE_REJECTED = False
frequency_rows = load_xy_sweep_samples(
    frequency_path,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
excitation_rows = load_xy_sweep_samples(
    excitation_path,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
{
    'frequency': {
        'rows': len(frequency_rows),
        'harmonic': f'h{xy_sweep_harmonic(frequency_rows)}',
    },
    'excitation': {
        'rows': len(excitation_rows),
        'harmonic': f'h{xy_sweep_harmonic(excitation_rows)}',
    },
}

## XY-only frequency scan

Only XY is shown. The legend and title state the harmonic order.

In [ ]:
for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
    figure = plot_xy_sweep(frequency_rows, metric=metric)
    display(figure)
    plt.close(figure)


## XY-only amplitude scan

The x axis is source RMS voltage; change it to `nominal_current_a_rms` only when that derived current is desired.

In [ ]:
for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
    figure = plot_xy_sweep(
        excitation_rows,
        metric=metric,
        x_axis='source_v_rms',
        log_x=False,
    )
    display(figure)
    plt.close(figure)


## Optional figure export

No files are written unless `SAVE_OUTPUTS=True`.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    for scan_name, rows in (
        ('frequency', frequency_rows),
        ('excitation', excitation_rows),
    ):
        for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
            figure = plot_xy_sweep(
                rows,
                metric=metric,
                x_axis='source_v_rms' if scan_name == 'excitation' else None,
                log_x=False if scan_name == 'excitation' else None,
            )
            figure.savefig(
                OUTPUT_DIRECTORY / f'{scan_name}_xy_{metric}.png', dpi=200
            )
            plt.close(figure)
    display(OUTPUT_DIRECTORY)
